# Week 12 Final Capstone: Customer Churn Prediction & Retention Strategy

This executed notebook follows the complete workflow from business framing and data validation through EDA, feature engineering, model tuning, interpretation, financial scenario analysis, and deployment demonstration.

**Business question:** Which customers are most likely to churn, and how should a retention team prioritize them?


## Phase 1 - Project setup and success metrics

Primary objective: identify churners early enough for a retention action. Because only 10.6% of customers churn, recall, F1, ROC-AUC, and PR-AUC are more informative than accuracy alone.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display
from src.config import RAW_DATA_PATH, SUCCESS_TARGETS
from src.data_validation import load_and_validate, quality_summary
from src.train import train_project
from src.predict import predict_customer

pd.set_option("display.max_columns", 30)
print("Success targets:")
print(json.dumps(SUCCESS_TARGETS, indent=2))


Success targets:
{
  "Recall": 0.8,
  "ROC_AUC": 0.85,
  "F1": 0.65,
  "Precision": 0.5
}


## Phase 2 - Data collection, dictionary, and quality validation


In [2]:
df = load_and_validate(RAW_DATA_PATH)
print(f"Validated shape: {df.shape}")
print(f"Churn customers: {df['Churn'].sum()} ({df['Churn'].mean():.1%})")
display(df.head())
display(quality_summary(df))


Validated shape: (500, 9)
Churn customers: 53 (10.6%)
  CustomerID  Tenure  MonthlyCharges  TotalCharges        Contract  \
0     C00001       6              64          1540        One year   
1     C00002      21             113          1753  Month-to-month   
2     C00003      27              31          1455        Two year   
3     C00004      53              29          7150  Month-to-month   
4     C00005      16             185          1023        One year   

      PaymentMethod PaperlessBilling  SeniorCitizen  Churn  
0       Credit Card               No              1      0  
1  Electronic Check              Yes              1      0  
2       Credit Card               No              1      0  
3  Electronic Check               No              1      0  
4  Electronic Check               No              1      0  
             Column Data_Type  Missing  Unique
0        CustomerID    object        0     500
1            Tenure     int64        0      71
2    MonthlyCharge

The supplied 500-row customer file is treated as an immutable raw snapshot. `CustomerID` is retained for traceability but excluded from training. The data dictionary is saved under `data/data_dictionary.csv`.


## Phase 3 - Exploratory data analysis


In [3]:
eda = (
    df.groupby("Contract")
      .agg(Customers=("CustomerID", "size"), ChurnRate=("Churn", "mean"),
           AverageTenure=("Tenure", "mean"), AverageMonthlyCharges=("MonthlyCharges", "mean"))
      .sort_values("ChurnRate", ascending=False)
)
display(eda.style.format({"ChurnRate": "{:.1%}", "AverageTenure": "{:.1f}", "AverageMonthlyCharges": "{:.1f}"}))
display(Image(filename="screenshots/02_eda_drivers.png", width=900))


<IPython.core.display.Image object>


Short tenure and month-to-month contracts emerge as important churn signals. These are predictive associations; they do not prove that changing a contract alone will cause retention.


## Phase 4 - Leakage-safe preprocessing, feature engineering, and model development


In [4]:
results = train_project()
print("Training complete.")
print("Selected model:", results["metadata"]["selected_model"])
print("Best parameters:", results["metadata"]["best_params"])
print("Decision threshold:", round(results["metadata"]["decision_threshold"], 3))


Training complete.
Selected model: Tuned Random Forest
Best parameters: {'model__max_depth': 6, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__n_estimators': 300}
Decision threshold: 0.38


The pipeline creates eight engineered features, imputes defensively, standardizes numeric variables, one-hot encodes categories, and tunes Random Forest hyperparameters with five-fold stratified cross-validation. Threshold selection uses only out-of-fold training probabilities.


In [5]:
comparison = pd.read_csv("outputs/model_comparison.csv")
display(comparison[["Model", "Threshold", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]].style.format("{:.3f}", subset=["Threshold", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]))
display(Image(filename="screenshots/03_model_comparison.png", width=900))


<IPython.core.display.Image object>


## Phase 5 - Evaluation, interpretation, and testing


In [6]:
metrics = results["metrics"]
key_metrics = {key: metrics[key] for key in ["Accuracy", "Balanced_Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "Specificity"]}
display(pd.Series(key_metrics, name="Holdout score").to_frame().style.format("{:.3f}"))
print("Success targets met:", results["metadata"]["success_targets_met"])
display(Image(filename="screenshots/04_confusion_matrix.png", width=520))
display(Image(filename="screenshots/05_roc_pr_curves.png", width=900))


Success targets met: {'Recall': True, 'ROC_AUC': True, 'F1': True, 'Precision': True}
<IPython.core.display.Image object>
<IPython.core.display.Image object>


In [7]:
importance = pd.read_csv("outputs/feature_importance.csv")
display(importance.head(12).style.format({"Importance": "{:.4f}"}))
display(Image(filename="screenshots/06_feature_importance.png", width=800))


<IPython.core.display.Image object>


Feature importance explains which variables the Random Forest used most, but it is not causal attribution. The model card documents limitations, responsible use, drift monitoring, and retraining requirements.


## Phase 6 - Business recommendations and impact scenario


In [8]:
impact = pd.read_csv("outputs/business_impact.csv")
display(impact.style.format({"Estimated_Net_Value_INR": "INR {:,.0f}"}))
display(Image(filename="screenshots/07_business_impact.png", width=800))


<IPython.core.display.Image object>


Recommended operating policy: prioritize High-risk customers for a human retention call, use lighter digital outreach for Medium risk, and avoid unnecessary discounts for Low risk. The financial values are transparent assumptions for scenario comparison, not realized revenue.


## Phase 7 - Deployment demonstration


In [9]:
sample = json.loads(Path("deployment/sample_request.json").read_text())
response = predict_customer(sample)
print("Sample request:")
print(json.dumps(sample, indent=2))
print("\nModel response:")
print(json.dumps(response, indent=2))


Sample request:
{
  "Tenure": 8,
  "MonthlyCharges": 180,
  "TotalCharges": 1400,
  "Contract": "Month-to-month",
  "PaymentMethod": "Electronic Check",
  "PaperlessBilling": "Yes",
  "SeniorCitizen": 1
}

Model response:
{
  "churn_probability": 0.9175,
  "prediction": 1,
  "risk_level": "High",
  "decision_threshold": 0.38,
  "recommended_action": "Priority retention call with a targeted offer."
}


In [10]:
low_risk = {
    "Tenure": 60, "MonthlyCharges": 65, "TotalCharges": 4800,
    "Contract": "Two year", "PaymentMethod": "Credit Card",
    "PaperlessBilling": "No", "SeniorCitizen": 0,
}
validation_examples = pd.DataFrame([predict_customer(sample), predict_customer(low_risk)], index=["High-risk example", "Lower-risk example"])
display(validation_examples)


                    churn_probability  prediction risk_level  \
High-risk example              0.9175           1       High   
Lower-risk example             0.0000           0        Low   

                    decision_threshold  \
High-risk example                 0.38   
Lower-risk example                0.38   

                                                   recommended_action  
High-risk example      Priority retention call with a targeted offer.  
Lower-risk example  Standard engagement; no retention discount nee...  


## Final conclusion

The project delivers a reproducible training pipeline, persisted model, batch evidence, FastAPI endpoint, Streamlit interface, Docker configuration, business recommendation, professional reports, presentation, tests, and career materials. A real deployment should add authentication, live-label monitoring, subgroup checks, and controlled retention experiments.
